# Stage 9b — What quantization actually costs

**Goal:** measure the quality/size tradeoff on *your* model, rather than
repeating folklore about it.

### The claim you'll read everywhere

> "Q4_K_M is the sweet spot — near-lossless at a quarter the size."

That advice is written for **7B+ models**, and for those it is roughly true. It
is not a law of nature. A larger model has enormous parameter redundancy: many
weights encode overlapping information, so rounding each one to 4 bits loses
little that the others can't cover.

A **15.7M-parameter** model has far less slack. Every weight is doing more work,
so the same rounding error costs more. We should expect Q4_K_M to hurt
noticeably here — and this notebook checks whether that's true instead of
assuming it.

This is a good habit generally: the quantization level is a decision you should
make from a measurement on your model, not from a table on the internet.

### What the levels mean

| level | bits/weight | scheme |
|---|---|---|
| f16 | 16 | no quantization — the reference |
| Q8_0 | 8 | one scale per block of 32 weights, uniform |
| Q5_K_M | ~5.5 | K-quant: sub-block scales, **mixed** — important tensors keep more bits |
| Q4_K_M | ~4.5 | K-quant, more aggressive |

The "K" quants beat flat schemes because they don't treat all tensors equally:
attention output and the feed-forward down-projection get higher precision,
since errors there propagate furthest.

In [ ]:
# --- Local bootstrap -------------------------------------------------------
# Kernel: "tinyllm (local, no torch)" -- registered by scripts\setup_local.ps1.
# There is deliberately no PyTorch in this environment.
import subprocess, sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from tinyllm import config
from tinyllm.config import model_cfg, quant_cfg, serve_cfg, gen_cfg, hub

MODELS = REPO / "models"
VENDOR = REPO / "vendor" / "llamacpp"

def llama_tool(name):
    """Locate a llama.cpp binary; its position inside the release zip moves."""
    hits = list(VENDOR.rglob(f"{name}.exe")) or list(VENDOR.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {VENDOR}. Run scripts\\setup_local.ps1.")
    return hits[0]

import importlib.util
assert importlib.util.find_spec("torch") is None, (
    "torch is installed in the local venv -- it should not be. See pyproject.toml."
)
print(f"repo:   {REPO}")
print(f"models: {[p.name for p in sorted(MODELS.glob('*.gguf'))] or 'none yet'}")
print("torch:  absent, by design")

## 9b.1 — Which quantizations do we have

In [ ]:
import subprocess, json, re, time

available = []
for q in ["f16"] + list(quant_cfg.levels):
    p = MODELS / f"{config.PROJECT_NAME}-{q}.gguf"
    if p.exists():
        available.append((q, p, p.stat().st_size / 1e6))

assert available, "No GGUF files. Run scripts\\pull_model.ps1 then scripts\\quantize.ps1."

base_mb = available[0][2]
print(f"{'level':<10} {'size MB':>10} {'vs f16':>8}")
print("-" * 30)
for q, p, mb in available:
    print(f"{q:<10} {mb:>10.2f} {mb / base_mb:>7.0%}")

## 9b.2 — Get held-out text to measure on

We need text the model never saw. Rather than downloading the 7.6 GB dataset,
pull a hundred validation rows from the Hub's datasets-server API — a few
hundred KB over HTTP.

Using the *validation* split matters. Measuring perplexity on training data
would tell you how well the model memorised, not how well it generalises, and
quantization damage would be partly hidden by that memorisation.

In [ ]:
import requests

txt_path = REPO / "models" / "perplexity_corpus.txt"

if not txt_path.exists():
    url = ("https://datasets-server.huggingface.co/rows"
           "?dataset=roneneldan%2FTinyStories&config=default&split=validation"
           "&offset=0&length=100")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    stories = [row["row"]["text"] for row in r.json()["rows"]]
    txt_path.write_text("\n\n".join(stories), encoding="utf-8")
    print(f"downloaded {len(stories)} held-out stories")

text = txt_path.read_text(encoding="utf-8")
print(f"{txt_path.name}: {len(text):,} chars, ~{len(text.split()):,} words")
print(f"\nfirst story:\n{text[:300]}...")

## 9b.3 — Measure perplexity at each level

`llama-perplexity` slides a window over the text and reports the exponentiated
mean cross-entropy. Because every level sees the identical text and identical
window size, the numbers are directly comparable.

This is CPU-bound on 2 cores — a couple of minutes per level.

In [ ]:
def measure_ppl(model_path, ctx=512, chunks=40, threads=serve_cfg.threads):
    """Run llama-perplexity and parse the final estimate."""
    exe = llama_tool("llama-perplexity")
    t0 = time.time()
    proc = subprocess.run(
        [str(exe), "-m", str(model_path), "-f", str(txt_path),
         "-c", str(ctx), "--chunks", str(chunks), "-t", str(threads)],
        capture_output=True, text=True,
    )
    out = proc.stdout + proc.stderr
    m = re.search(r"Final estimate:\s*PPL\s*=\s*([0-9.]+)\s*\+/-\s*([0-9.]+)", out)
    if not m:
        print(out[-1500:])
        raise RuntimeError(f"could not parse perplexity for {model_path.name}")
    return float(m.group(1)), float(m.group(2)), time.time() - t0

results = []
for q, p, mb in available:
    print(f"measuring {q} ...", end=" ", flush=True)
    ppl, err, secs = measure_ppl(p)
    results.append({"quant": q, "size_mb": mb, "ppl": ppl, "err": err, "secs": secs})
    print(f"PPL {ppl:.4f} +/- {err:.4f}   ({secs:.0f}s)")

In [ ]:
base = results[0]["ppl"]
print(f"\n{'level':<10} {'size MB':>9} {'vs f16':>8} {'PPL':>9} {'delta':>9} {'% worse':>9}")
print("-" * 60)
for r in results:
    d = r["ppl"] - base
    print(f"{r['quant']:<10} {r['size_mb']:>9.2f} {r['size_mb']/results[0]['size_mb']:>7.0%} "
          f"{r['ppl']:>9.4f} {d:>+9.4f} {100*d/base:>8.2f}%")

## 9b.4 — The tradeoff curve

In [ ]:
import matplotlib.pyplot as plt

quants = [r["quant"] for r in results]
sizes = [r["size_mb"] for r in results]
ppls = [r["ppl"] for r in results]
errs = [r["err"] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.errorbar(sizes, ppls, yerr=errs, marker="o", markersize=9,
            color="#4C78A8", linewidth=2, capsize=4)
for q, s, p in zip(quants, sizes, ppls):
    ax.annotate(q, (s, p), textcoords="offset points", xytext=(8, 6), fontsize=10)
ax.axhline(base, color="#54A24B", linestyle=":", label="f16 reference")
ax.set_xlabel("file size (MB)"); ax.set_ylabel("perplexity (lower is better)")
ax.set_title("Quality vs size")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
degr = [100 * (p - base) / base for p in ppls]
bars = ax.bar(quants, degr, color=["#54A24B" if d < 1 else "#F58518" if d < 5 else "#E45756"
                                   for d in degr])
ax.bar_label(bars, fmt="%+.2f%%", padding=3, fontsize=9)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("perplexity increase vs f16 (%)")
ax.set_title("Cost of quantization")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout(); plt.show()

## 9b.5 — Does the number match what you can read?

Perplexity is a proxy. Generate from each level with the same seed and greedy
decoding, and judge for yourself whether the measured degradation shows up as
visibly worse text.

Sometimes it doesn't, and that's informative too: a 3% perplexity increase you
can't perceive is a very different decision from a 3% increase that produces
obvious grammatical breakdown.

In [ ]:
PROMPT = "Once upon a time, there was a little rabbit who"

def gen(model_path, n=60, temp=0.0, seed=0):
    exe = llama_tool("llama-cli")
    proc = subprocess.run(
        [str(exe), "-m", str(model_path), "-p", PROMPT, "-n", str(n),
         "--temp", str(temp), "--seed", str(seed), "-no-cnv",
         "--no-warmup", "-t", str(serve_cfg.threads)],
        capture_output=True, text=True,
    )
    return " ".join(proc.stdout.strip().split())

for r in results:
    p = MODELS / f"{config.PROJECT_NAME}-{r['quant']}.gguf"
    print(f"=== {r['quant']}  (PPL {r['ppl']:.3f}, {r['size_mb']:.1f} MB)")
    print(f"  {gen(p)}\n")

## 9b.6 — Speed

On a 2-core CPU, smaller weights also mean **fewer bytes to move**, and
inference at this size is memory-bandwidth bound rather than compute bound. So
quantization usually buys latency as well as disk.

In [ ]:
def bench(model_path, n=100):
    exe = llama_tool("llama-cli")
    t0 = time.time()
    subprocess.run(
        [str(exe), "-m", str(model_path), "-p", PROMPT, "-n", str(n),
         "--temp", "0", "-no-cnv", "--no-warmup", "-t", str(serve_cfg.threads)],
        capture_output=True, text=True,
    )
    return n / (time.time() - t0)

print(f"{'level':<10} {'tok/s':>8}")
print("-" * 20)
speeds = []
for r in results:
    p = MODELS / f"{config.PROJECT_NAME}-{r['quant']}.gguf"
    s = bench(p)
    speeds.append(s)
    print(f"{r['quant']:<10} {s:>8.1f}")

print("\n(Rough numbers -- includes process startup, and this is a 2-core CPU.)")

## What to conclude

Fill this in from *your* numbers, not from the table you read somewhere:

- **Q8_0** should be nearly indistinguishable from f16. At roughly half the size,
  it is the safe default, and it's what `serve.ps1` uses.
- **Q4_K_M** is where you should look hardest. If the degradation here is larger
  than the "negligible" you were promised, you've just measured the thing this
  notebook exists to show: **quantization advice does not transfer across model
  scales.**

The general lesson is about method, not about this model. Quantization level is a
decision to make from a measurement on the model you actually have.

## Stage 9b gate

- [x] Perplexity measured on held-out text at every level
- [x] Quality/size curve plotted
- [x] Numbers checked against readable output
- [x] Throughput measured

**Next:** `11_client_api.ipynb`